### What is RAG?

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes — in many cases you can still join after the course has started, but it depends on the course’s enrollment policy and whether there’s still space.

A few things to check:
- whether late enrollment is allowed
- the course start/end dates
- any prerequisite or placement requirements
- whether you’ll miss required early work

If you want, I can help you draft a quick message to the instructor or registrar asking if it’s still possible to enroll.


In [7]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#How should I start the course and follow the weekly workflow?
Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

Watch the lesson videos.
Work through the lesson notebooks/code.
Read the homework instructions on GitHub.
Submit answers through the course platform before the deadline.
Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the required peer reviews. Homework is not required. You can work through the material and prepare your project in self-paced mode, but project submission and peer review must happen while a live cohort is accepting them.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
If the homework title contains [DRAFT], it means the homework is not ready yet.

The homework is ready only when both are true:

The homework form is open on the course management platform.
The homework title does not contain [DRAFT].
Until then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.
'''

In [9]:
prompt = f"""
Your task is to answer questions from the course participants based on the context provided. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}

"""

In [10]:
print(prompt)


Your task is to answer questions from the course participants based on the context provided. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors

In [12]:
question = "I just discovered the course. Can I still join?"
answer = llm(question)
print(answer)

Yes — usually you can still join if enrollment is open or if the instructor allows late entry.

A quick way to check:
1. Look at the course page for enrollment status or deadlines.
2. Contact the instructor or course admin and ask if late registration is possible.
3. If there’s a waitlist, ask whether seats are likely to open up.

If you want, I can help you draft a short message asking to join late.


In [19]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

### Dataset

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [17]:
documents[500]

{'id': 'b75fa5280e',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': 'How do I install extra packages on Google Colab or Kaggle?',
 'answer': "In a notebook cell, prefix `pip install` with `!`:\n\n```bash\n!pip install tensorflow[and-cuda]==2.14\n```\n\nFor packages with extras you want installed silently, use `-q`:\n\n```bash\n!pip install -q xgboost==2.1.0\n```\n\nRestart the runtime if a package overrides one that's already imported (Runtime → Restart runtime)."}

In [18]:
documents[1000]["question"]

'How set Pandas to show entire text content in a column. Useful to view the entire Explanation column content in the LLM-as-judge section of the offline-rag-evaluation notebook'

### Search from the KB (Knowledge Base)